# Validation — `kappa-lora-metamathqa-top50`

**What this measures:** `num_trainable_params` is emitted by the repo's own MetaMathQA→GSM8K harness (`run.py`) and directly measures the "halves trainable params vs standard LoRA" half of the claim against the published `lora--llama-3.2-3B-rank32` row (9,175,040): with `condition_number_top_fraction=0.5` exactly 28 of the 56 q/v modules are adapted, so the value must land in 3,670,016–5,505,024 (40–60% of baseline; exactly 50% iff 14 q_proj + 14 v_proj are selected), while `test_accuracy` guards the "without losing fit" half at the user's notebook ±0.02 parity band. The new experiment config mirrors the published row verbatim (same base model, r=32, alpha=64, dropout 0.0, [v_proj, q_proj]) plus only the new knob — `r`/`lora_alpha` are retained because the PR extends `LoraConfig` itself (they are the class's own fields, not unknown keys), so the single delta between arm and comparator is the spectral-targeting switch, invoked exactly as in the user's Colab verification (`python run.py --verbose --clean <experiment-dir>`, compare vs reference with PASS/INVESTIGATE bands).

**How I read the claim:** The PR adds a kappa-LoRA-style knob: rank the modules matched by target_modules by the condition number of their base weights and inject LoRA into only the top fraction. Under this repo's published protocol the candidate set is just 56 modules (28 q_proj at 196,608 LoRA params each, 28 v_proj at 131,072 each, summing to the row's 9,175,040), so the top-50% cut selects 28 modules and yields 40-60% of baseline parameters - exactly 50% (4,587,520) only if 14 q and 14 v are chosen, which the notebook must print. The verification notebook (per user_guidance, cloned from the Colab pattern) runs the new experiments/kappa_lora config through method_comparison/MetaMathQA/run.py and compares against the published lora--llama-3.2-3B-rank32 row. Support = num_trainable_params at most 5,505,024 and on the lattice 3,670,016 + k*65,536, test_accuracy at least 0.4705 (row 0.4905 minus the repo's 0.02 parity band, 2 seeds since the arm genuinely differs from baseline), total_time within +3 min of 1173.7 s and memory not above 22,286,434,304 bytes. Caveat: the paper's halving was over all LoRA matrices and its -16.2% time / -4.5% memory come from a different setting; with q/v-only targeting the time/memory savings will likely undershoot those numbers, so they are printed as expectations, not gates.

- ⚠️ The paper halves over the full population of LoRA-eligible weight matrices; this protocol's candidates are only q_proj+v_proj in two unequal shapes, so 'top 50% of modules' is 40-60% of parameters, exactly 50% only for a 14q+14v split. The claim must be tested as a count/threshold, not a fraction.
- ⚠️ Paper's -16.2% time and -4.5% memory are averages over its own benchmarks with larger target sets; here adapters are 9,175,040 of 3,221,924,864 params (0.285%) on q/v only, so the LoRA compute slice is small and savings will likely undershoot the paper, while the selection pre-pass adds 56 SVDs (largest 3072x3072) of upfront cost.
- ⚠️ Accuracy-matching was established on the paper's tasks/models, not MetaMathQA -> GSM8K on Llama-3.2-3B at r=32/alpha=64; it transfers only as a hypothesis under this row's protocol.

**Target metric:** `num_trainable_params`

**Repository:** [mayorquinmachines/peft](https://github.com/mayorquinmachines/peft) at commit [`57dd468527da`](https://github.com/mayorquinmachines/peft/commit/57dd468527da38edead8878e9c9b262e2f4c1684)

**Benchmark:** the repository's own `method_comparison/MetaMathQA/run.py` over `experiments/lora/llama-3.2-3B-rank32-kappa-top50` — not a synthesized stand-in, so the numbers are comparable to what this repository publishes.

**Nothing here has been executed** — there are no outputs and no result is being claimed. Review the measurement, edit the configuration or criteria if it is wrong, then mention `@remyx validate` to run it on Remyx compute — or run the cells top to bottom yourself on a machine with a GPU.

In [ ]:
# Parameters (Remyx passes the commit it measures as `ref`)
variant = "feature"
ref = ""
seed = 0

## 1. Environment

A CUDA GPU is required; the published protocol peaks above 22 GB.

In [ ]:
!nvidia-smi -L
import sys, torch
print(f"python {sys.version.split()[0]} · torch {torch.__version__} · cuda {torch.cuda.is_available()}")

## 2. The code under test

Clone the repository and check out exactly the commit that was validated, then install it in editable mode so the harness imports this checkout. When this notebook runs on Remyx compute the checkout already exists at that commit, and this cell only confirms it.

In [ ]:
import os, subprocess, sys
REPO_URL = "https://github.com/mayorquinmachines/peft"
COMMIT = ref or "57dd468527da38edead8878e9c9b262e2f4c1684"

def _sh(*cmd):
    return subprocess.run(cmd, check=True, text=True, capture_output=True).stdout.strip()

def _at_commit():
    try:
        return os.path.isdir(".git") and _sh("git", "rev-parse", "HEAD").startswith(COMMIT)
    except Exception:
        return False

if not _at_commit():
    if not os.path.isdir("repo"):
        _sh("git", "clone", "--quiet", REPO_URL, "repo")
    os.chdir("repo")
    _sh("git", "fetch", "--quiet", "--depth=1", "origin", COMMIT)
    _sh("git", "checkout", "--quiet", COMMIT)
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", "."], check=True)
ROOT = os.getcwd()
print(ROOT)
print(_sh("git", "log", "-1", "--oneline"))

## 3. Credentials

If the benchmark downloads gated models or datasets it needs a Hugging Face token. In Colab, store it as a secret named `HF_TOKEN`; elsewhere set the environment variable.

In [ ]:
import os
if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        pass
print("HF_TOKEN set" if os.environ.get("HF_TOKEN") else "HF_TOKEN not set — gated downloads will fail")

## 4. The experiment configuration

The harness runs a method by its configuration directory. This validation points it at `experiments/lora/llama-3.2-3B-rank32-kappa-top50` (relative to `method_comparison/MetaMathQA`).

In [ ]:
import glob
for path in sorted(glob.glob(os.path.join(ROOT, "method_comparison/MetaMathQA/experiments/lora/llama-3.2-3B-rank32-kappa-top50/*"))):
    print(f"--- {path} ---")
    print(open(path).read())

## 5. Confirm the change under test is what is loaded

The commit printed here must match the one checked out above.

In [ ]:
import importlib
print(_sh("git", "rev-parse", "HEAD"))

## 6. Run the benchmark

`method_comparison/MetaMathQA/run.py` over `experiments/lora/llama-3.2-3B-rank32-kappa-top50` — a directory of experiments runs each in turn; a single experiment runs once.

In [ ]:
os.chdir(os.path.join(ROOT, "method_comparison/MetaMathQA"))
import glob, importlib, runpy, sys, time
RUN_STARTED = time.time()
configs = sorted(glob.glob("experiments/lora/llama-3.2-3B-rank32-kappa-top50/*/")) or ["experiments/lora/llama-3.2-3B-rank32-kappa-top50"]
for cfg in configs:
    print(f"[remyx] {cfg}")
    sys.argv = ["run.py", cfg.rstrip("/")]
    runpy.run_path("run.py", run_name="__main__")

## 7. Read what the benchmark wrote

Results land under `temporary_results/lora--*.json` (relative to `method_comparison/MetaMathQA`) — or wherever this harness writes for a non-default checkout; only a document written by the run above counts. The metrics the criteria are judged against are fields of that document.

In [ ]:
import glob, json, os
PATTERNS = ["temporary_results/lora--*.json"]
paths = sorted((p for pat in PATTERNS for p in glob.glob(pat)), key=os.path.getmtime)
paths = [p for p in paths if os.path.getmtime(p) >= RUN_STARTED - 1]
if not paths:
    # Some harnesses write elsewhere depending on the checkout (peft uses
    # temporary_results/ off the main branch): any document this run wrote.
    paths = sorted((p for p in glob.glob("**/*.json", recursive=True)
                    if os.path.getmtime(p) >= RUN_STARTED - 1 and "experiments/" not in p),
                   key=os.path.getmtime)
assert paths, "the benchmark wrote no result document"
doc = json.load(open(paths[-1]))
print("result document:", paths[-1])

def find(obj, key):
    """Last value under `key` anywhere in the document ('test accuracy' matches test_accuracy)."""
    hit = None
    if isinstance(obj, dict):
        for k, v in obj.items():
            if str(k).replace(" ", "_") == key and isinstance(v, (int, float)):
                hit = v
            found = find(v, key)
            hit = found if found is not None else hit
    elif isinstance(obj, list):
        for item in obj:
            found = find(item, key)
            hit = found if found is not None else hit
    return hit

METRICS = ["num_trainable_params", "test_accuracy", "total_time", "accelerator_memory_max"]
observed = {name: find(doc, name) for name in METRICS}
print(json.dumps(observed, indent=2))

## 8. Against the criteria

Thresholds come from `.remyx/validation.yaml`, so a failing measurement reports rather than crashes. `baseline` is the published row this repository already ships for the comparison method.

In [ ]:
CRITERIA = [
    {
        "metric": "num_trainable_params",
        "direction": "<=",
        "threshold": 5505024,
        "baseline": 9175040.0
    },
    {
        "metric": "test_accuracy",
        "direction": ">=",
        "threshold": 0.4705,
        "baseline": 0.49052312357846856
    },
    {
        "metric": "total_time",
        "direction": "<=",
        "threshold": 1353.7,
        "baseline": 1173.714544863964
    },
    {
        "metric": "accelerator_memory_max",
        "direction": "<=",
        "threshold": 22286434304,
        "baseline": 22286434304.0
    }
]

print(f"{'metric':<28}{'observed':>16}{'baseline':>16}  criterion")
for c in CRITERIA:
    v = observed.get(c["metric"])
    t = c["threshold"]
    ok = None if v is None or t is None else (v <= t if c["direction"] == "<=" else v >= t)
    mark = "?" if ok is None else ("PASS" if ok else "FAIL")
    fmt = lambda x: (f"{x:.6g}" if isinstance(x, float) else str(x))
    print(f"{c['metric']:<28}{fmt(v):>16}{fmt(c['baseline']):>16}  {c['direction']} {fmt(t)}  {mark}")

## 9. Report

One line, machine-readable — what Remyx records as this run's measurement.

In [ ]:
print(json.dumps(observed))

## 10. What the outcome means

- **All rows pass** → the claim holds at this protocol: `num_trainable_params` <= 5505024 with `test_accuracy` >= 0.4705, `total_time` <= 1353.7, `accelerator_memory_max` <= 22286434304 holding.
- **`num_trainable_params` fails** → the change does not deliver what the claim says at this protocol.
- **A guardrail fails** → the target may be met at the cost of something the claim promised to keep; look at the run log before drawing a conclusion.
- **No result document** → the benchmark did not finish; the run cell above says why.

## Appendix — the criteria file

`.remyx/validation.yaml` as committed:

```yaml
model:
  provider: zai
loop: {max_iterations: 8, fix_code: true}
benchmarks:
  - name: kappa-lora-metamathqa-top50
    suite:
      harness:
        runner: "method_comparison/MetaMathQA/run.py"
        experiments: "experiments/lora/llama-3.2-3B-rank32-kappa-top50"
        results_glob: "method_comparison/MetaMathQA/temporary_results/lora--*.json"
        method: "lora"
        smoke:
          params_path: "method_comparison/MetaMathQA/default_training_params.json"
          overrides:
            num_steps: 8
            eval_steps: 4
      scorer: "num_trainable_params"
      metrics:
        - name: "num_trainable_params"
          direction: min
          # 28 of 56 matched modules kept (ceil(0.5*56)); worst case all q_proj: 28*32*(3072+3072)=5,505,024 = 60.0% of baseline 9,175,040.
          # Expected value = 3,670,016 + k*65,536 with k = number of selected q_proj in [0,28]; exactly 50% (4,587,520) iff k=14.
          threshold: 5505024
          role: target
        - name: "test_accuracy"
          direction: max
          # 0.49052312357846856 - 0.02 = one-sided floor at the repo's demonstrated parity band on this exact harness/model/eval (user_resource notebook).
          threshold: 0.4705
          role: guardrail
        - name: "total_time"
          direction: min
          # row total_time 1173.714544863964 s + the notebook's 3-min (180 s) wall-clock tolerance = 1353.71 s.
          threshold: 1353.7
          role: cost
        - name: "accelerator_memory_max"
          direction: min
          # no memory regression vs the row's 22,286,434,304 bytes (paper's -4.5% = 21,283,544,760 is an expectation, not a gate).
          threshold: 22286434304
          role: cost
      policy:
        guardrail_veto: true
    baseline:
      source: "method_comparison/MetaMathQA/results/lora--llama-3.2-3B-rank32.json"
      values:
        num_trainable_params: 9175040.0
        num_total_params: 3221924864.0
        test_accuracy: 0.49052312357846856
        total_time: 1173.714544863964
        train_time: 958.3276851720293
        accelerator_memory_max: 22286434304.0
        forgetting: 0.4156990051269531
    compute:
      tier: gpu
      # one arm: row total_time 1173.7s + 180s tolerance + gated Llama-3.2-3B download (~6.5 GB) + 56-SVD pre-pass (seconds, largest 3072x3072) + setup/eval margin
      timeout_s: 5400
    held_constant:
      - "base model meta-llama/Llama-3.2-3B, r=32, lora_alpha=64, lora_dropout=0.0, target_modules [v_proj, q_proj] copied verbatim from the published row config so the only delta is condition_number_top_fraction"
      - "5000 train steps and GSM8K test eval from the harness default protocol (default_training_params.json) — no training_params.json override in the experiment dir"
      - "harness invocation python run.py --verbose --clean experiments/lora/llama-3.2-3B-rank32-kappa-top50, the same pattern as the user's verification notebook"
      - "same seed and data ordering from harness defaults, single run, no sweep"
    avoid:
      - "no unpinned base-model revision — the harness default revision is the protocol"
      - "do not gate on bit-exactness; adapter-init RNG differs run to run, so accuracy is judged only within the ±0.02 band"
      - "do not write into results/ — the published corpus is read-only; PR-branch runs land in temporary_results/"
      - "do not claim a fixed 50.0% param cut — selection is by module count (28 of 56), so params land in 40-60% depending on the q/v kappa split; report the realized fraction"
    provenance:
      num_trainable_params: "user_guidance"
      test_accuracy: "user_guidance + user_resource:https://colab.research.google.com/drive/1z73-jtAGrq4HkjvorjFcZ77uMwdmWs56?usp=sharing (±0.02 parity band)"
      total_time: "user_resource:https://colab.research.google.com/drive/1z73-jtAGrq4HkjvorjFcZ77uMwdmWs56?usp=sharing (±3 min tolerance) anchored on published_row:method_comparison/MetaMathQA/results/lora--llama-3.2-3B-rank32.json"
      accelerator_memory_max: "published_row:method_comparison/MetaMathQA/results/lora--llama-3.2-3B-rank32.json"
      baseline: "published_row:method_comparison/MetaMathQA/results/lora--llama-3.2-3B-rank32.json"
      suite: "repo_runner:method_comparison/MetaMathQA/run.py"
      experiments: "inferred from PR diff field condition_number_top_fraction on LoraConfig; protocol values from published_row:method_comparison/MetaMathQA/results/lora--llama-3.2-3B-rank32.json"
      smoke: "inferred from sibling training_params.json naming (5k steps, lr) — key names correctable by fix_code iterations"
      held_constant: "protocol_doc:method_comparison/MetaMathQA/README.md + published row config"
```